In [ ]:
#Flask
#3rd party library
#create web services
#Web service --data is exposed in form of url


In [ ]:
pip install flask

In [ ]:
from flask import Flask,request
import math
app =Flask(__name__)
@app.route("/")
def hello():
    return ("Hello World")
@app.route("/welcome")
def welcome():
    return "Welcome to the world of Flask"
@app.route("/wu/<user>")#Path param ---input data is passed in url
def welcome1(user):
    return "Welcome {}".format(user)
@app.route("/sqrt/<int:n>")
def findsqrt(n):
    return "SQRT is {}".format(math.sqrt(n))
@app.route("/add/<float:n1>/<float:n2>")
def add(n1,n2):
    return "Sum of {} and {} is {}".format(n1,n2,n1+n2)
#http://localhost:5000/sub?n1=100&n2=20---query param
@app.route("/sub")
def sub():
    n1 =int(request.args.get("n1"))
    n2 =int(request.args.get("n2"))
    return "Diff of {} and {} is {}".format(n1,n2,n1-n2)  
@app.route("/posteg",methods=["POST"])
def posteg():
    data=request.get_json()
    print("Received",data)
    return data
    
app.run()

In [ ]:
from flask import Flask,request,jsonify
app =Flask(__name__)
students={}
@app.route("/liststudents")
def liststudents():
    return jsonify(students)
    
@app.route("/addmodifystudent",methods=["POST"])
def addstudent():
    data =request.get_json()
    rollNo=data.get("rollNo")
    students[rollNo]=data
    return jsonify({"success":True})
    
@app.route("/delete/<rollNo>",methods=["DELETE"])
def deletestudent(rollNo):
    rollNo=int(rollNo)
    print("rollNo",rollNo)
    try:
        del students[rollNo]
        return jsonify({"success":True})
    except Exception as e:
       print("Exception ",e)
       return jsonify({"success":False,"error":"Student not found"})
    
app.run()


In [1]:
import threading

In [2]:
def hello():
    print("Hello from thread")

In [7]:
t1=threading.Thread(target=(hello))

In [8]:
t1.start()

Hello from thread


In [9]:
t2=threading.Thread(target=(hello))

In [10]:
t2.start()

Hello from thread


In [11]:
#Thread pool is a pool od reusable threads

In [12]:
from concurrent.futures import ThreadPoolExecutor

In [14]:
with ThreadPoolExecutor(max_workers=3) as executor:
    executor.submit(hello)
    executor.submit(hello)
    executor.submit(hello)
    executor.submit(hello)
    executor.submit(hello)


Hello from threadHello from thread

Hello from thread
Hello from thread
Hello from thread


In [7]:
import time
import threading
from concurrent.futures import ThreadPoolExecutor
def newhello():
    time.sleep(2)
    print("Hello from",threading.current_thread().name)
    
with ThreadPoolExecutor(max_workers=3) as executor:
    executor.submit(newhello)
    executor.submit(newhello)
    executor.submit(newhello)
    executor.submit(newhello)
    executor.submit(newhello)


Hello from ThreadPoolExecutor-3_0
Hello from ThreadPoolExecutor-3_2
Hello from ThreadPoolExecutor-3_1
Hello from ThreadPoolExecutor-3_2
Hello from ThreadPoolExecutor-3_0


In [8]:
import sqlite3
conn = sqlite3.connect("test.db")
conn.execute(""" 
create table iris( sepallength text,	sepalwidth text,	petallength text,	petalwidth text,	variety text)
""")
conn.close()

In [11]:
def insert(dataToInsert):
    threadName=threading.current_thread().name
    print(threadName,"processing")
    conn = sqlite3.connect("test.db")
    cursor =conn.cursor()
    sqlinsert="insert into iris(sepallength,sepalwidth,petallength,petalwidth,variety) values(?,?,?,?,?)"
    cursor.execute(sqlinsert,dataToInsert)
    conn.commit()
    conn.close()
    print(threadName,"done")


In [13]:
import csv

In [14]:
with ThreadPoolExecutor(max_workers=3) as executor:
    with open("C://t//test//iris.csv","r") as ip:
        dr=csv.DictReader(ip)
        for row in dr:
            dataToInsert=(row['sepal.length'],row['sepal.width'],row['petal.length'],row['petal.width'],row['variety'])
            executor.submit(insert,dataToInsert)

ThreadPoolExecutor-4_0 processing
ThreadPoolExecutor-4_1 processing
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_2 done
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_2 done
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_2 done
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_1 done
ThreadPoolExecutor-4_1 processing
ThreadPoolExecutor-4_1 done
ThreadPoolExecutor-4_1 processing
ThreadPoolExecutor-4_2 done
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_1 done
ThreadPoolExecutor-4_1 processing
ThreadPoolExecutor-4_0 done
ThreadPoolExecutor-4_0 processing
ThreadPoolExecutor-4_0 done
ThreadPoolExecutor-4_0 processing
ThreadPoolExecutor-4_2 done
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_2 done
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_2 done
ThreadPoolExecutor-4_2 processing
ThreadPoolExecutor-4_1 done
ThreadPoolExecutor-4_1 processing
ThreadPoolExecutor-4_1 done
ThreadPoolExecutor-4_1 processing
ThreadPoolExecutor-4_1 done
Th

In [16]:
import sqlite3
conn = sqlite3.connect("test.db")
conn.row_factory=sqlite3.Row
cursor =conn.cursor()

cursor.execute("select count(*) from iris")
rows=cursor.fetchall()
for row in rows:
    print(dict(row))
conn.close()

{'count(*)': 150}


In [19]:
#LOCKS are basically to ensure only one thread get access to critical section
import threading
from concurrent.futures import ThreadPoolExecutor
counter=0
lock=threading.Lock()
def increment():
    threadName=threading.current_thread().name
    print(threadName,"waiting")
    global counter
    lock.acquire()
    print(threadName,"got lock")
    counter=counter+1
    lock.release()
    print(threadName,"released lock","counter",counter)
    
import csv

In [20]:
with ThreadPoolExecutor(max_workers=3) as tp:
    with open("C://t//test//iris.csv","r") as ip:
        dr =csv.DictReader(ip)
        for row in dr:
            tp.submit(increment)

ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 1
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 2
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 3
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 4
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 5
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 6
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 7
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 8
ThreadPoolExecutor-5_0 waiting
ThreadPoolExecutor-5_0 got lock
ThreadPoolExecutor-5_0 released lock counter 9
ThreadPool

In [21]:
#LOCKS are basically to ensure only one thread get access to critical section
import threading
from concurrent.futures import ThreadPoolExecutor
counter=0
lock=threading.Lock()
def incrementWithoutLock():
    threadName=threading.current_thread().name
    global counter
    counter=counter+1
    print(threadName,"released lock","counter",counter)
    
import csv

In [22]:
with ThreadPoolExecutor(max_workers=3) as tp:
    with open("C://t//test//iris.csv","r") as ip:
        dr =csv.DictReader(ip)
        for row in dr:
            tp.submit(incrementWithoutLock)

ThreadPoolExecutor-6_0 released lock counter 1
ThreadPoolExecutor-6_0 released lock counter 2
ThreadPoolExecutor-6_0 released lock counter 3
ThreadPoolExecutor-6_0 released lock counter 4
ThreadPoolExecutor-6_0 released lock counter 5
ThreadPoolExecutor-6_0 released lock counter 6
ThreadPoolExecutor-6_0 released lock counter 7
ThreadPoolExecutor-6_0 released lock counter 8
ThreadPoolExecutor-6_0 released lock counter 9
ThreadPoolExecutor-6_0 released lock counter 10
ThreadPoolExecutor-6_0 released lock counter 11
ThreadPoolExecutor-6_0 released lock counter 12
ThreadPoolExecutor-6_0 released lock counter 13
ThreadPoolExecutor-6_0 released lock counter 14
ThreadPoolExecutor-6_0 released lock counter 15
ThreadPoolExecutor-6_0 released lock counter 16
ThreadPoolExecutor-6_0 released lock counter 17
ThreadPoolExecutor-6_0 released lock counter 18
ThreadPoolExecutor-6_0 released lock counter 19
ThreadPoolExecutor-6_0 released lock counter 20
ThreadPoolExecutor-6_0 released lock counter 21
T

In [23]:
 pip install parameterized

  Obtaining dependency information for parameterized from https://files.pythonhosted.org/packages/00/2f/804f58f0b856ab3bf21617cccf5b39206e6c4c94c2cd227bde125ea6105f/parameterized-0.9.0-py2.py3-none-any.whl.metadata
Using cached parameterized-0.9.0-py2.py3-none-any.whl (20 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
pip install bs4

  Obtaining dependency information for bs4 from https://files.pythonhosted.org/packages/51/bb/bf7aab772a159614954d84aa832c129624ba6c32faa559dfb200a534e50b/bs4-0.0.2-py2.py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import requests
from bs4 import BeautifulSoup

In [27]:
wikidata=requests.get("https://en.wikipedia.org/wiki/India").text

In [30]:
mydoc=BeautifulSoup(wikidata,"html.parser")

In [33]:
all_anchors=mydoc.find_all("a")
for a in all_anchors:
    print(a.text,a.attrs.get("href"))

Jump to content #bodyContent
Main page /wiki/Main_Page
Contents /wiki/Wikipedia:Contents
Current events /wiki/Portal:Current_events
Random article /wiki/Special:Random
About Wikipedia /wiki/Wikipedia:About
Contact us //en.wikipedia.org/wiki/Wikipedia:Contact_us
Help /wiki/Help:Contents
Learn to edit /wiki/Help:Introduction
Community portal /wiki/Wikipedia:Community_portal
Recent changes /wiki/Special:RecentChanges
Upload file /wiki/Wikipedia:File_upload_wizard
Special pages /wiki/Special:SpecialPages






 /wiki/Main_Page

Search
 /wiki/Special:Search
Donate https://donate.wikimedia.org/?wmf_source=donate&wmf_medium=sidebar&wmf_campaign=en.wikipedia.org&uselang=en
Create account /w/index.php?title=Special:CreateAccount&returnto=India
Log in /w/index.php?title=Special:UserLogin&returnto=India
Donate https://donate.wikimedia.org/?wmf_source=donate&wmf_medium=sidebar&wmf_campaign=en.wikipedia.org&uselang=en
 Create account /w/index.php?title=Special:CreateAccount&returnto=India
 Log in /

In [34]:
alltables=mydoc.find_all("table")
for t in alltables:
    print(t)

<table class="infobox ib-country vcard"><tbody><tr><th class="infobox-above adr" colspan="2"><div class="fn org country-name">Republic of India</div><div class="ib-country-names"><span title="ISO 15919 Indic (Hindi language) transliteration"><i lang="hi-Latn">Bhārat Gaṇarājya</i></span></div></th></tr><tr><td class="infobox-image" colspan="2"><div class="noresize" style="display:table; width:100%;">
<div style="display:table-cell; vertical-align:middle; padding-left:5px;">
<div style="padding-bottom:3px;"><span class="mw-image-border" typeof="mw:File"><a class="mw-file-description" href="/wiki/File:Flag_of_India.svg" title="Flag of India"><img alt="Horizontal tricolour flag bearing, from top to bottom, deep saffron, white, and green horizontal bands. In the centre of the white band is a navy-blue wheel with 24 spokes." class="mw-file-element" data-file-height="600" data-file-width="900" decoding="async" height="83" src="//upload.wikimedia.org/wikipedia/en/thumb/4/41/Flag_of_India.svg/2

In [35]:
import pandas as pd

In [36]:
alltables=pd.read_html("https://en.wikipedia.org/wiki/India")

In [38]:
i =1
for table in alltables:
    table.to_excel("C://t//testing//"+str(i)+".xlsx")
    i=i+1